# Phase 0a — test trên Colab T4 (miễn phí)

Chạy lần lượt từng ô. Runtime → **Change runtime type → T4 GPU**.

| Ô | Kiểm tra | Chạy được trên T4? |
| --- | --- | --- |
| 1–2 | GPU, code repo (test + demo 3D) | Có |
| 3–5 | Ảnh phòng thật → MoGe-2 → kích thước phòng → cảnh 3D tải về xem | Có (MoGe-2 nhẹ, 331M) |
| 6 | TRELLIS (bản 1), SAM 3, Qwen-Image-Edit GGUF | **Chưa kiểm chứng** — xem ghi chú cuối |

Colab free: ~16 GB VRAM, ~12–13 GB RAM (số liệu từ blog bên thứ ba, không phải Google công bố), phiên có thể bị ngắt.

In [ ]:
# 1) GPU + RAM
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h | head -2

In [ ]:
# 2) Lấy code (repo private → dán token GitHub chỉ-đọc, không hiện trên màn hình), chạy test + demo 3D
import getpass, os, subprocess
REPO = "/content/AI_Agent"
if not os.path.exists(REPO):
    token = getpass.getpass("GitHub token (chỉ-đọc, repo AI_Agent): ")
    subprocess.run(["git", "clone", "-q", "--branch", "develop",
                    f"https://{token}@github.com/Tan-Phat-20KDL/AI_Agent.git", REPO], check=True)
    del token
    # xoá token khỏi cấu hình git của bản clone
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/Tan-Phat-20KDL/AI_Agent.git"], check=True)
%cd /content/AI_Agent
!pip -q install trimesh pytest
!python -m pytest -q
!python tools/demo_local.py

In [ ]:
# 3) Tải 1 ảnh phòng thật lên (thấy rõ sàn)
from google.colab import files
up = files.upload()
ROOM = list(up)[0]
print("Ảnh phòng:", ROOM)

In [ ]:
# 4) MoGe-2: ảnh → point map mét + normal → mask sàn (theo hướng ngửa lên) → kích thước phòng
!pip -q install git+https://github.com/microsoft/MoGe.git
import sys, cv2, torch, numpy as np
sys.path.insert(0, "/content/AI_Agent")
from moge.model.v2 import MoGeModel
from core.room import estimate_room, floor_mask_from_normals

dev = "cuda"
model = MoGeModel.from_pretrained("Ruicheng/moge-2-vitl-normal").to(dev)
img = cv2.cvtColor(cv2.imread(ROOM), cv2.COLOR_BGR2RGB)
scale = 1600 / max(img.shape[:2])
if scale < 1:  # ảnh quá lớn → thu nhỏ cho đỡ tốn VRAM
    img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
x = torch.tensor(img / 255, dtype=torch.float32, device=dev).permute(2, 0, 1)
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    out = model.infer(x)
pts = out["points"].cpu().numpy()
valid = out["mask"].cpu().numpy().astype(bool)
floor = floor_mask_from_normals(out["normal"].cpu().numpy(), valid)
print(f"VRAM đỉnh MoGe-2: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB | điểm sàn: {floor.mean():.0%} ảnh")

REAL_MM = None   # vd 2800 nếu biết trần cao 2.8 m → chạy lại ô này để hiệu chỉnh tỉ lệ
est = estimate_room(pts, floor)
if REAL_MM:
    est = estimate_room(pts, floor, calibration=(est.height_mm, REAL_MM))
print(f"Phòng ước lượng: rộng {est.width_mm} × sâu {est.depth_mm} × cao {est.height_mm} mm | "
      f"camera cao {est.camera_height_mm} mm | sàn khớp mặt phẳng {est.floor_inlier_ratio:.0%}")

In [ ]:
# 5) Dựng cảnh 3D với kích thước phòng vừa đo (bàn/ghế vẫn là khối tạm) → tải zip về máy xem bằng viewer
from core.placement import PlacementError
from core.scene3d import build_scene, table_chairs_spec
spec = table_chairs_spec("colab", (est.width_mm, est.depth_mm, est.height_mm),
                         "runtime/assets/table.glb", [1600, 900, 750], "runtime/assets/chair.glb", [450, 520, 820], 4,
                         texture_path="runtime/inputs/wood.png", texture_on="floor", tile_mm=(1200, 200))
try:
    build_scene(spec, ".", "runtime/out/colab/scene.glb")
    !zip -qr /content/colab_scene.zip runtime/out/colab runtime/inputs/wood.png runtime/assets
    files.download("/content/colab_scene.zip")
    print("Giải nén vào thư mục repo trên máy bạn, rồi mở viewer với ?scene=runtime/out/colab/scene.glb")
except PlacementError as e:
    print("Phòng ước lượng quá nhỏ cho 1 bàn + 4 ghế (chỉ thấy một phần sàn?). Đặt REAL_MM ở ô 4 rồi chạy lại.
", e)

In [ ]:
# 5b) Mở giao diện Gradio với link công khai (bấm link *.gradio.live hiện ra bên dưới)
!pip -q install gradio
!python app/gradio_app.py --share


## 6) Bước sau — CHƯA kiểm chứng chạy được trên T4

| Model | Vì sao chưa chắc | Nguồn |
| --- | --- | --- |
| TRELLIS (bản 1, MIT) — ảnh bàn/ghế → GLB | README yêu cầu GPU **≥ 16 GB**; T4 có ~15–16 GB → sát giới hạn. Chỉ test trên Linux, A100/A6000 | [github.com/microsoft/TRELLIS](https://github.com/microsoft/TRELLIS) |
| TRELLIS.2 | Yêu cầu **≥ 24 GB** → không chạy trên T4 | [github.com/microsoft/TRELLIS.2](https://github.com/microsoft/TRELLIS.2) |
| SAM 3 (848M) | Nhẹ; cần CUDA ≥ 12.6, PyTorch ≥ 2.7 — kiểm tra bản Colab đang có | [github.com/facebookresearch/sam3](https://github.com/facebookresearch/sam3) |
| Qwen-Image-Edit-2511 | Bản bf16/FP8 quá lớn cho T4. Bản GGUF Q4 (cộng đồng) có thể vừa VRAM nhưng hướng dẫn cộng đồng khuyên **≥ 32 GB RAM**, Colab chỉ ~12 GB → khả năng cao không chạy | [docs ComfyUI 2511](https://docs.comfy.org/tutorials/image/qwen/qwen-image-edit-2511) |

Kết quả ô 1–5 (VRAM đỉnh, kích thước phòng so với đo tay) ghi vào `docs/SRS.md` mục Phase 0.